<a href="https://colab.research.google.com/github/swatiraiwani/Customer-Churn-Prediction-Using-Machine-Learning/blob/main/End_to_End_NLP_Project_Sentiment_Analysis_on_Product_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 0: Initial Setup and Library Installations
!pip install -q pandas numpy scikit-learn nltk gensim tensorflow keras transformers datasets streamlit pyngrok

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# For Keras/TensorFlow
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

# For Hugging Face Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# For Streamlit (later)
import streamlit as st
from pyngrok import ngrok

# Download NLTK resources (run this once)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Setup complete. Libraries installed and imported.")

Setup complete. Libraries installed and imported.


In [ ]:
# Step 1: Data Loading and Preprocessing

# Sample Data (Replace with your actual dataset if you have one)
data = {
    'review_text': [
        "This product is amazing! I love it.",
        "Absolutely terrible. Waste of money.",
        "It's okay, not great but not bad either.",
        "I would highly recommend this to everyone.",
        "The quality is poor and it broke after a day.",
        "Best purchase I've made this year!",
        "Not satisfied with this item. Disappointed.",
        "Works as expected. Good value.",
        "Could be better, but for the price, it's fine.",
        "Fantastic! Will buy again.",
        "Horrible experience, do not buy.",
        "I'm very happy with this product.",
        "It's a decent product, nothing special.",
        "Worst product ever. Regret buying it.",
        "Good quality and fast shipping.",
        "Mediocre at best. Wouldn't recommend.",
        "Excellent value for money. Five stars!",
        "Broke easily. Not durable at all.",
        "I use it every day. Very useful.",
        "So-so. I've had better."
    ],
    'sentiment': [1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0] # 1 for positive, 0 for negative
}
df = pd.DataFrame(data)

print("Original DataFrame:")
print(df.head())
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

# Text Preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = re.sub(r'<[^>]+>', '', text) # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation and numbers
    text = text.lower() # Convert to lowercase
    words = text.split() # Tokenize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words] # Lemmatize and remove stopwords
    return ' '.join(words)

df['processed_text'] = df['review_text'].apply(preprocess_text)

print("\nDataFrame after preprocessing:")
print(df[['review_text', 'processed_text', 'sentiment']].head())

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

print(f"\nTraining samples: {len(X_train)}, Testing samples: {len(X_test)}")

Original DataFrame:
                                     review_text  sentiment
0            This product is amazing! I love it.          1
1           Absolutely terrible. Waste of money.          0
2       It's okay, not great but not bad either.          1
3     I would highly recommend this to everyone.          1
4  The quality is poor and it broke after a day.          0

Sentiment distribution:
sentiment
1    12
0     8
Name: count, dtype: int64

DataFrame after preprocessing:
                                     review_text  \
0            This product is amazing! I love it.   
1           Absolutely terrible. Waste of money.   
2       It's okay, not great but not bad either.   
3     I would highly recommend this to everyone.   
4  The quality is poor and it broke after a day.   

                    processed_text  sentiment  
0             product amazing love          1  
1  absolutely terrible waste money          0  
2            okay great bad either          1  
3  wou

In [ ]:
# Step 2: Method 1 - TF-IDF + Logistic Regression

# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=1000) # Consider top 1000 features/words
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nShape of TF-IDF training data: {X_train_tfidf.shape}")
print(f"Shape of TF-IDF testing data: {X_test_tfidf.shape}")

# Train Logistic Regression model
lr_model = LogisticRegression(solver='liblinear', random_state=42) # liblinear is good for small datasets
lr_model.fit(X_train_tfidf, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

# Evaluation
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"\nLogistic Regression Accuracy: {accuracy_lr:.4f}")
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

# Example prediction
sample_review_processed = preprocess_text("This is a fantastic product, I really love it!")
sample_review_tfidf = tfidf_vectorizer.transform([sample_review_processed])
prediction = lr_model.predict(sample_review_tfidf)
print(f"\nPrediction for '{sample_review_processed}': {'Positive' if prediction[0] == 1 else 'Negative'}")


Shape of TF-IDF training data: (16, 53)
Shape of TF-IDF testing data: (4, 53)

Logistic Regression Accuracy: 0.5000
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.50      1.00      0.67         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4


Prediction for 'fantastic product really love': Positive


In [ ]:
# Step 3: Method 2 - LSTM with Keras Embedding Layer

# Parameters for Keras
VOCAB_SIZE = 5000  # Max number of words to keep, based on frequency
MAX_LENGTH = 100   # Max length of a sequence (review)
EMBEDDING_DIM = 64 # Dimension of the dense embedding

# Tokenize text data
keras_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<unk>") # <unk> for out-of-vocabulary words
keras_tokenizer.fit_on_texts(X_train)

# Convert text to sequences of integers
X_train_sequences = keras_tokenizer.texts_to_sequences(X_train)
X_test_sequences = keras_tokenizer.texts_to_sequences(X_test)

# Pad sequences to ensure uniform length
X_train_padded = pad_sequences(X_train_sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

print(f"\nShape of padded training sequences: {X_train_padded.shape}")
print(f"Shape of padded testing sequences: {X_test_padded.shape}")

# Build the LSTM model
lstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LENGTH),
    Bidirectional(LSTM(64, return_sequences=True)), # Using Bidirectional LSTM
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("\nLSTM Model Summary:")
lstm_model.summary()

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
# Note: With our tiny dataset, this will likely overfit or not perform well.
# It's for demonstration purposes.
print("\nTraining LSTM model...")
history_lstm = lstm_model.fit(
    X_train_padded, y_train,
    epochs=15, # Increased epochs, but early stopping will help
    batch_size=4, # Small batch size for small dataset
    validation_data=(X_test_padded, y_test),
    callbacks=[early_stopping],
    verbose=1 # Set to 1 or 2 to see progress, 0 for silent
)

# Evaluate the model
loss_lstm, accuracy_lstm = lstm_model.evaluate(X_test_padded, y_test, verbose=0)
print(f"\nLSTM Model Test Accuracy: {accuracy_lstm:.4f}")

# Predictions
y_pred_lstm_probs = lstm_model.predict(X_test_padded)
y_pred_lstm = (y_pred_lstm_probs > 0.5).astype(int).flatten()

print("LSTM Model Classification Report:")
print(classification_report(y_test, y_pred_lstm, zero_division=0))

# Example prediction
sample_review_processed = preprocess_text("This is a fantastic product, I really love it!")
sample_review_sequence = keras_tokenizer.texts_to_sequences([sample_review_processed])
sample_review_padded = pad_sequences(sample_review_sequence, maxlen=MAX_LENGTH, padding='post', truncating='post')
prediction_prob = lstm_model.predict(sample_review_padded)[0][0]
prediction = 1 if prediction_prob > 0.5 else 0
print(f"\nPrediction for '{sample_review_processed}': {'Positive' if prediction == 1 else 'Negative'} (Prob: {prediction_prob:.2f})")


Shape of padded training sequences: (16, 100)
Shape of padded testing sequences: (4, 100)

LSTM Model Summary:


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training LSTM model...
Epoch 1/15


InvalidArgumentError: Graph execution error:

Detected at node sequential_2_1/bidirectional_2_1/backward_lstm_2_1/CudnnRNNV3 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File

In [ ]:
# Step 4: Method 3 - Hugging Face Transformers (Fine-tuning DistilBERT)

# Define model name
MODEL_NAME = 'distilbert-base-uncased' # A good, smaller BERT variant

# Load tokenizer
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Re-split data if necessary, or use the same X_train, X_test, y_train, y_test
# For Hugging Face, we typically work with raw text before our custom preprocessing,
# as the tokenizer handles sub-word tokenization, case, etc.
# Let's re-split from the original df, but use the 'review_text' (original)
X_train_hf_raw, X_test_hf_raw, y_train_hf, y_test_hf = train_test_split(
    df['review_text'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

# Convert to lists for the tokenizer
train_texts = X_train_hf_raw.tolist()
test_texts = X_test_hf_raw.tolist()
train_labels = y_train_hf.tolist()
test_labels = y_test_hf.tolist()

# Tokenize the texts
# The Hugging Face tokenizer will handle padding and truncation
train_encodings = hf_tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = hf_tokenizer(test_texts, truncation=True, padding=True, max_length=128)

# Create Hugging Face Dataset objects
class SentimentDataset(torch.utils.data.Dataset): # PyTorch dataset
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

import torch # Ensure torch is imported if not already

train_dataset_hf = SentimentDataset(train_encodings, train_labels)
test_dataset_hf = SentimentDataset(test_encodings, test_labels)

# Load pre-trained model for sequence classification
# num_labels=2 because we have positive (1) and negative (0)
hf_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Training Arguments
# Due to the extremely small dataset, we'll use minimal training steps.
# In a real scenario, you'd use more epochs/steps.
training_args = TrainingArguments(
    output_dir='./results_hf',
    num_train_epochs=3, # Small number of epochs for tiny dataset
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50, # Reduced warmup steps
    weight_decay=0.01,
    logging_dir='./logs_hf',
    logging_steps=1, # Log very frequently for small dataset
    evaluation_strategy="epoch", # Evaluate at the end of each epoch
    save_strategy="epoch",       # Save model at the end of each epoch
    load_best_model_at_end=True, # Load the best model found during training
    report_to="none" # Suppress wandb/tensorboard reporting for this small example
)

# Define a compute_metrics function for evaluation
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

# Trainer
trainer = Trainer(
    model=hf_model,
    args=training_args,
    train_dataset=train_dataset_hf,
    eval_dataset=test_dataset_hf,
    compute_metrics=compute_metrics
)

# Fine-tune the model
print("\nFine-tuning Hugging Face Transformer model...")
trainer.train()

# Evaluate the fine-tuned model
eval_results = trainer.evaluate()
print(f"\nHugging Face Model Evaluation Results: {eval_results}")
accuracy_hf = eval_results['eval_accuracy']
print(f"Hugging Face Model Test Accuracy: {accuracy_hf:.4f}")

# Example prediction
sample_review_raw = "This is a fantastic product, I really love it!"
inputs = hf_tokenizer(sample_review_raw, return_tensors="pt", truncation=True, padding=True)

# Move inputs to the same device as the model (CPU or GPU)
# If using GPU, model is already on GPU. If CPU, inputs on CPU.
# trainer.model.device will tell you where the model is.
inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

with torch.no_grad(): # Disable gradient calculations for inference
    logits = hf_model(**inputs).logits

predicted_class_id = torch.argmax(logits, dim=-1).item()
print(f"\nPrediction for '{sample_review_raw}': {'Positive' if predicted_class_id == 1 else 'Negative'}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# Step 5: Model Saving & Loading (Hugging Face Model)

# The Trainer already saved the best model if load_best_model_at_end=True.
# The path to the best model is usually trainer.state.best_model_checkpoint
# Let's explicitly save the final trained model (which should be the best one here)
# to a specific directory for clarity.

SAVE_PATH_HF = "./my_sentiment_classifier_hf"
hf_model.save_pretrained(SAVE_PATH_HF)
hf_tokenizer.save_pretrained(SAVE_PATH_HF)
print(f"\nHugging Face model and tokenizer saved to {SAVE_PATH_HF}")

# To load the model and tokenizer later:
loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH_HF)
loaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH_HF)
print("\nLoaded Hugging Face model and tokenizer successfully.")

# Test loaded model
sample_review_raw = "This is a terrible product, I hate it."
inputs = loaded_tokenizer(sample_review_raw, return_tensors="pt", truncation=True, padding=True)

# Ensure model and inputs are on the same device (e.g., CPU for this loading example)
# If you trained on GPU and are loading on CPU, map location:
# loaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH_HF, map_location=torch.device('cpu'))
# loaded_model.to('cpu') # Ensure model is on CPU
# inputs = {k: v.to('cpu') for k, v in inputs.items()}

# If you have a GPU available and want to use it:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}


with torch.no_grad():
    logits = loaded_model(**inputs).logits
predicted_class_id = torch.argmax(logits, dim=-1).item()

print(f"\nPrediction using loaded model for '{sample_review_raw}': {'Positive' if predicted_class_id == 1 else 'Negative'}")

# To save the TF-IDF model (vectorizer + classifier):
import joblib
TFIDF_MODEL_PATH = './tfidf_logreg_model.joblib'
joblib.dump({'vectorizer': tfidf_vectorizer, 'classifier': lr_model}, TFIDF_MODEL_PATH)
print(f"\nTF-IDF + Logistic Regression model saved to {TFIDF_MODEL_PATH}")

# To load TF-IDF model:
# loaded_tfidf_model_components = joblib.load(TFIDF_MODEL_PATH)
# loaded_tfidf_vectorizer = loaded_tfidf_model_components['vectorizer']
# loaded_lr_classifier = loaded_tfidf_model_components['classifier']
# print("TF-IDF model loaded.")

# To save the Keras LSTM model:
KERAS_MODEL_PATH = './keras_lstm_model.h5'
lstm_model.save(KERAS_MODEL_PATH)
# Save the Keras tokenizer
import pickle
KERAS_TOKENIZER_PATH = './keras_tokenizer.pkl'
with open(KERAS_TOKENIZER_PATH, 'wb') as handle:
    pickle.dump(keras_tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print(f"\nKeras LSTM model saved to {KERAS_MODEL_PATH} and tokenizer to {KERAS_TOKENIZER_PATH}")

# To load Keras model:
# from tensorflow.keras.models import load_model
# loaded_keras_model = load_model(KERAS_MODEL_PATH)
# with open(KERAS_TOKENIZER_PATH, 'rb') as handle:
#    loaded_keras_tokenizer = pickle.load(handle)
# print("Keras model and tokenizer loaded.")


Hugging Face model and tokenizer saved to ./my_sentiment_classifier_hf

Loaded Hugging Face model and tokenizer successfully.



Prediction using loaded model for 'This is a terrible product, I hate it.': Negative

TF-IDF + Logistic Regression model saved to ./tfidf_logreg_model.joblib

Keras LSTM model saved to ./keras_lstm_model.h5 and tokenizer to ./keras_tokenizer.pkl


In [ ]:
%%writefile app.py
# Step 6: Deployment with Streamlit (Bonus)
# This is the content of app.py

import streamlit as st
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import re # For basic preprocessing if needed, though HF tokenizer handles most

# --- Configuration ---
MODEL_PATH = "./my_sentiment_classifier_hf" # Path where your fine-tuned model is saved
# If running locally and model is not in current dir, provide full path.

# --- Load Model and Tokenizer ---
# Add error handling for model loading
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    # Ensure model is in evaluation mode
    model.eval()
    # Move model to GPU if available, else CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model_loaded_successfully = True
except Exception as e:
    st.error(f"Error loading the model or tokenizer: {e}")
    model_loaded_successfully = False

# --- Prediction Function ---
def predict_sentiment(text):
    if not model_loaded_successfully:
        return "Model not loaded.", 0.0

    # Basic text cleaning (optional, as HF tokenizer is robust)
    # text = re.sub(r'<[^>]+>', '', text) # Remove HTML
    # text = text.strip()

    if not text:
        return "Please enter some text.", 0.0

    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to the same device as model

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        probabilities = torch.softmax(logits, dim=-1)
        predicted_class_id = torch.argmax(probabilities, dim=-1).item()
        confidence = probabilities[0][predicted_class_id].item()

        sentiment = "Positive" if predicted_class_id == 1 else "Negative"
        return sentiment, confidence
    except Exception as e:
        st.error(f"Error during prediction: {e}")
        return "Error in prediction.", 0.0

# --- Streamlit App Interface ---
st.set_page_config(page_title="Sentiment Analyzer", layout="wide")

st.title("📝 Product Review Sentiment Analyzer")
st.markdown("""
    Enter a product review below, and the AI will predict whether the sentiment is positive or negative.
    This model was fine-tuned using Hugging Face Transformers (DistilBERT).
""")

if not model_loaded_successfully:
    st.warning("The sentiment analysis model could not be loaded. Please check the logs.")
else:
    st.sidebar.header("About")
    st.sidebar.info(
        "This app uses a DistilBERT model fine-tuned for sentiment classification. "
        "It can classify text into 'Positive' or 'Negative' categories."
    )

    st.header("Enter Review Text:")
    user_input = st.text_area("Type your review here...", height=150, key="review_input")

    if st.button("Analyze Sentiment", key="analyze_button"):
        if user_input:
            with st.spinner("Analyzing..."):
                sentiment, confidence = predict_sentiment(user_input)

            if sentiment == "Positive":
                st.success(f"Sentiment: **{sentiment}** (Confidence: {confidence:.2f})")
            elif sentiment == "Negative":
                st.error(f"Sentiment: **{sentiment}** (Confidence: {confidence:.2f})")
            else: # Error or no input
                st.warning(sentiment) # Display error message from predict_sentiment
        else:
            st.warning("Please enter some text to analyze.")

st.markdown("---")
st.markdown("Built with [Streamlit](https://streamlit.io) & [Hugging Face](https://huggingface.co).")

Writing app.py


In [ ]:
# Step 6 (Continued): Run Streamlit app with ngrok

# Kill any existing ngrok tunnels if you rerun this cell
!killall ngrok

# Set your ngrok authtoken (replace with your actual token from ngrok.com dashboard)
# You only need to do this once per Colab session, or if your token changes.
# Get a free token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN" # <--- REPLACE THIS WITH YOUR TOKEN
if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTHTOKEN":
    print("Please replace 'YOUR_NGROK_AUTHTOKEN' with your actual ngrok authtoken.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Run the Streamlit app in the background
    # Make sure 'app.py' and the model directory 'my_sentiment_classifier_hf' are in the root of your Colab environment.
    # If not, adjust paths in app.py accordingly.
    !nohup streamlit run app.py --server.port 8501 &

    # Open a tunnel to the Streamlit app
    try:
        public_url = ngrok.connect(8501)
        print("\nStreamlit App is live at:")
        print(public_url)
        print("\nIf you see 'connection refused', wait a few seconds for Streamlit to start and then refresh the ngrok URL.")
    except Exception as e:
        print(f"Error starting ngrok: {e}")
        print("Make sure you have set your NGROK_AUTH_TOKEN correctly and that ngrok is not already running a tunnel on this port.")

ngrok: no process found
Please replace 'YOUR_NGROK_AUTHTOKEN' with your actual ngrok authtoken.


In [ ]:
!killall ngrok
!kill $(ps aux | grep 'streamlit run app.py' | awk '{print $2}')
print("Streamlit app and ngrok should be stopped.")

ngrok: no process found
^C
Streamlit app and ngrok should be stopped.
